In [1]:
exec(open("_0_fns.py",encoding="utf-8").read())
my_aiff  ='/Users/yerik/Music/_1_NEW_SOURCE/_2023_this/_23_04_Foundation_Hotel'

<string>:209: DeprecationWarning: 'aifc' is deprecated and slated for removal in Python 3.13


In [2]:
"""
===========================
PREPROCESSING MODULE
===========================
"""
# --- LOAD LIBRARY ---
exec(open("_0_fns.py",encoding="utf-8").read())# Uncomment to set your AIFF source directory:
# my_aiff = "/Volumes/MUSIC_PROD/_1_NEW_SOURCE copy"
# my_aiff = "/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/cover"

# --- CONFIGURE ENVIRONMENT ---
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- PRE-PROCESSING & ANALYSIS ---
results_df = analyze_and_downsize_aiff(my_aiff)
print(results_df.head())
print('\nALL AIFF in correct format -> if something changes go erase the already downsized\n')
input('ENTER to analyze \n')

# --- FILE FILTERING & ATTRIBUTE EXTRACTION ---
code = 'all_AIFF'
audio_extensions = ['.AIFF', '.aif', '.aiff']
file_extensions = audio_extensions
my_folder_path = my_aiff
table_name = f"temp_now_{code}.pkl"

df, fre_tab = _filefinder_by_EXT_GET_df(my_folder_path, file_extensions=audio_extensions)
df = df[~df['Name'].str.startswith('._')].copy()
df = df[~df['Name'].str.startswith('.DS')].copy()
df = df.rename(columns={'Name': 'temp_id'}).drop(columns=['Size (MB)',])
df = df[df['Extension'].isin(['.aiff', '.AIFF'])].copy()

# -----------------------------------------------------------------
# SETUP FOLDER & TABLE VARIABLES
# -----------------------------------------------------------------
my_folder_path = my_aiff
table_name = f"temp_now_{code}.pkl"

# -----------------------------------------------------------------
# SECTION 1: READ & FILTER AUDIO FILES
# -----------------------------------------------------------------
df, fre_tab = _filefinder_by_EXT_GET_df(my_folder_path, file_extensions=audio_extensions)  # file_extensions = None -> 4 none ext
df = df[~df['Name'].str.startswith('._')].copy()
df = df[~df['Name'].str.startswith('.DS')].copy()  # Omit unwanted files
df = df.rename(columns={'Name': 'temp_id'}).drop(columns=['Size (MB)',])  # Keep & rename columns
df = df[df['Extension'].isin(['.aiff', '.AIFF'])].copy()  # Keep only AIFF files

# -----------------------------------------------------------------
# SECTION 2: EXTRACT GENERAL AUDIO ATTRIBUTES
# -----------------------------------------------------------------
df = _audio_attr_extract_11_INFO_AIFF(df, audio_extensions)

# -----------------------------------------------------------------
# SECTION 3: LUFS CALCULATIONS & CATEGORIZATION
# -----------------------------------------------------------------
df['ms_lufs'] = compute_lufs_for_paths_AIFF(df['Path'])
df['ms_LUFS_code'] = df['ms_lufs'].apply(_all_values_CREATE_12_LUFS_categories)
lufs_cols = ['ms_lufs', 'ms_LUFS_code']
df = _lufs_1004_i1_GET_df_id_cat_lufs(df)

# -----------------------------------------------------------------
# SECTION 4: BPM DYNAMIC VS NORMAL ANALYSIS
# -----------------------------------------------------------------
df = _df_bpm_2409_i1_GET_df_bpm_variation(
    df,
    path_column='Path',
    sr_column='sr',
    exclude_start_pct=0.20,
    exclude_end_pct=0.10
)
df = _cat_0204_bpm_consistency_GET_cat(df)

# -----------------------------------------------------------------
# SECTION 5: METADATA PROCESSING - TITLE HANDLING
# -----------------------------------------------------------------
df = _title_0204_id3_filefallback_GET_df_with_title(df, 'title', 'TRkw', 'ARkw')
print('DONE with getting 1_title')
print(df['title_file'].value_counts())
df = _col_2409_txt_clean_single_GET_df(df, 'title')
_write_title_id3_bulk(df)
# =============================================================================
# SECTION: METADATA PROCESSING - _2_artist
# =============================================================================
df = df = _artist_0204_id3_filefallback_GET_df_with_artist(df, 'artist', 'ARkw', 'MXkw')
print('DONE with getting 2_artist')
print(df['artist_file'].value_counts())
df = _col_2409_txt_clean_single_GET_df(df, 'artist')
_write_artist_id3_bulk(df)

# =============================================================================
# SECTION: METADATA PROCESSING - _3_LABEL
# =============================================================================
df = _label_0204_id3_filefallback_GET_df_with_LABEL(df, 'LABEL', 'LBkw', 'RYkw')
print('DONE with getting 3_LABEL')
print(df['label_file'].value_counts())
df['LABEL'] = df['LABEL'].str.replace('_', ' ').str.replace(r'^[^a-zA-Z]+', '', regex=True)

# =============================================================================
# SECTION: METADATA PROCESSING - _4_GENRE
# =============================================================================
df = _genre_0204_id3_filefallback_GET_df_with_genre(df, 'genre', 'GNkw', 'RMkw')
print('DONE with getting 4_genre')
print(df['genre_file'].value_counts())
df['genre'] = df['genre'].str.replace('_', ' ').str.replace(r'^[^a-zA-Z]+', '', regex=True)

# =============================================================================
# SECTION: METADATA PROCESSING - _5_release_year
# =============================================================================
df = _relyear_0204_id3_filefallback_GET_df_with_rel_year(df, 'rel_year', 'YRkw', 'PYkw')
print('DONE with getting 5_release_year')
print(df['rel_year_file'].value_counts())

# =============================================================================
# SECTION: METADATA PROCESSING - _6_KEY
# =============================================================================
df = _key_0204_id3_filefallback_GET_df_with_key(df, 'KEY', 'KYkw', 'BPkw')
print('DONE with getting 6_KEY')
print(df['key_file'].value_counts())
df['KEY'] = df['KEY'].str.replace('_', ' ').str.replace(r'^[^a-zA-Z]+', '', regex=True)

# =============================================================================
# SECTION: METADATA PROCESSING - Remixer and Mix Type (_7_mix_name & _8_remixer)
# =============================================================================
df = _mixremix_0204_filename_extract_GET_df_mix_and_remixer(df)
print('DONE with getting 7_mix_name & 8_remixer')
df[['mix_name', 'remixer']].head()
from tqdm.auto import tqdm; tqdm.pandas()
df['remix'] = df['Path'].progress_apply(lambda x: 'R' if 'remix' in str(x).lower() else 'U')

# =============================================================================
# SECTION: METADATA PROCESSING - Purchased Date (_9_date_purchased)
# =============================================================================
df = _datepurch_0204_filename_extract_GET_df_with_date_purchased(df)
print('DONE with getting 9_date_purchased')

# =============================================================================
# SECTION: WRITE ID3 TAGS - Remixer & Mix Name
# =============================================================================
df['remixer'] = (df['mix_name'].fillna('') + ' ' + df['remixer'].fillna('')).str.strip()
df['remixer'] = df['remixer'].str.replace('_', ' ').str.replace(r'^[^a-zA-Z]+', '', regex=True)
_aiff_0102_i1_GET_update_remixer_tpe4_tag(df)
print('DONE with WRITTING 7_mix_name & 8_remixer : to ID3TAG remixer')

# =============================================================================
# SECTION: WRITE ID3 TAGS - Genre
# =============================================================================
processed_count = _write_genre_id3_bulk(df, path_col="Path", genre_col="genre")
print('DONE with WRITTING 4_genre : to ID3TAG genre')

# =============================================================================
# SECTION: WRITE ID3 TAGS - Label
# =============================================================================
processed_label_count = _write_label_id3_bulk(df, path_col="Path", label_col="LABEL")
print('DONE with WRITTING 3_LABEL : to ID3TAG LABEL')

# =============================================================================
# SECTION: METADATA PROCESSING - Release Date
# =============================================================================
df = _reldate_0204_id3_filefallback_GET_df_with_rel_date(df, 'rel_date', 'RYkw_', '_PYkw')
_write_reldate_id3_bulk(df, path_col='Path', reldate_col='rel_date')

# =============================================================================
# SECTION: KEY PROCESSING & ADDITIONAL TAGS
# =============================================================================
#exec(open("_fns_key.py", encoding="utf-8").read())
df = _key_bulk_0815_librosa_middle_aiff_GET_output(df)
df = _key_0403_i2_GET_keys(df)
df = _write_tags_2712_id3_SET_key_bulk(df, "Path", "key_dj")
df = _mix_0804_i1_GET_df_5cols(df)

# =============================================================================
# SECTION: UPDATE ID & COMMENT
# =============================================================================
df['ID'] = ('dy' + df['bpm_consistency_cat'].str[2] 
            + df['ms_LUFS_code'].str[2].str.upper() 
            + df['id_cat_lufs'].str.lower() +'_'
            + df['title'].str[0].str.lower()
            + df['artist'].str[0].str.upper()
            + df['file_size'].astype(str).str[:2])
df['ms_lufs']
df.sort_values(by='ID')
df['comment'] = df['ID'] +'_'+ df['KEY']
_write_comment_id3_bulk(df)

# =============================================================================
# SECTION: RENAME FILES
# =============================================================================
# df['re_name'] = ( df['ID'] +'_'
#                  +'BY_'+ df['artist'].astype(str).str[:5]
#                  +'_'+df['title'].astype(str).str[:9]
#                  +'_'+ df['remix'].astype(str))
# _rename_aiff_files_bulk(df)

# =============================================================================
# SECTION: TEXT CLEANING (Assumed external call)
# =============================================================================
# cleaned_df = _col_2409_txt_clean_GET_df(df, list_of_columns)

# =============================================================================
# SECTION: HASH COMPUTATION
# =============================================================================
df = _hash_bulk_2812_audio_GET_df_hashes(df)
print("DOoooooooooooo\n\n\n\n\n\n\nne...")

# =============================================================================
# SECTION: FINAL EXECUTION
# =============================================================================
DO IT



   Sample Rate (Hz)  Bit Depth  Count
0             44100         16      9

ALL AIFF in correct format -> if something changes go erase the already downsized



ENTER to analyze 
 


Searching for Files: 100%|████████████████████████████████████████████████| 9/9 [00:00<00:00, 10714.94files/s]


ATTN ::: <E>  AFTER making sure that you have assigned the DIRECTORY PATH to ::: var ::: /Users/yerik/Music/_1_NEW_SOURCE/_2023_this/_23_04_Foundation_Hotel
***df*** will be returned having ::: 9  rows, write df in next cell to see your DATA FRAME

second part FREQUENCIES OF df :::

Grand Total Size in MB: 494.586
Grand Total Size in Bytes: 518611108.000
Grand Total Size in GB: 0.483

 FREQUENCY TABLE::: 

  Extension  frequency  total_size_in_mb
0     .aiff          9        494.586094


Searching for Files: 100%|████████████████████████████████████████████████| 9/9 [00:00<00:00, 13627.70files/s]


...

Extracting ID3 Titles: 100%|███████████████████████████████████████████████████| 9/9 [00:00<00:00, 373.72it/s]


DONE with getting 1_title
title_file
ID3TAGS    9
Name: count, dtype: int64


Writing AIFF files for title: 0it [00:00, ?it/s]

Total files processed for title: 0


0